
# C6-pytorch — Session 3: Parameter Counting and Inspection

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2 —
in particular the pinned `DenseLayer` / `ThresholdGate` modules, the
`nn.Parameter(..., requires_grad=False)` registration discipline, and
submodule composition.*

**This session:** a network you can build, you must also be able to
*audit*.
How many numbers does it hold?
Where do they live, under what names, with what shapes?
The exam asks these questions directly — usually with calculator-free
arithmetic and sometimes with its favorite twist, "compute the count
**without** calling the helper that computes it."
Today: the `parameters()` family, counting by hand, `numel` and the
count-without-`numel` register, `state_dict` inspection, a worked
normal-form MC, and solving *backwards* from a count to an
architecture.


In [ ]:

import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention


class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


def zeros_dense(out_units, in_units):
    """A DenseLayer of the given size with all-zero placeholder numbers."""
    return DenseLayer(torch.zeros(out_units, in_units), torch.zeros(out_units))



## 1. The `parameters()` Family

**Motivation.**
Session 2 registered numbers so torch could see them; this session uses
the seeing.
Every `nn.Module` offers:

- **`m.parameters()`** — iterates over every registered parameter
  tensor, including those of all submodules, recursively;
- **`m.named_parameters()`** — the same walk, yielding
  `(dotted_name, tensor)` pairs;
- **`m.state_dict()`** — the same contents as a dictionary
  `{dotted_name: tensor}` (Section 4).

The model on the bench for the whole session: a $3 \to 4 \to 2$
network, dense–gate–dense–gate.


In [ ]:

bench = nn.Sequential(
    zeros_dense(4, 3),      # 3 inputs -> 4 hidden units
    ThresholdGate(),
    zeros_dense(2, 4),      # 4 hidden -> 2 outputs
    ThresholdGate(),
)

n_tensors = 0
for name, p in bench.named_parameters():
    print(f"{name:10s} shape {tuple(p.shape)}  requires_grad={p.requires_grad}")
    n_tensors += 1
print("parameter tensors:", n_tensors)



Four parameter *tensors* — `0.weight (4, 3)`, `0.bias (4,)`,
`2.weight (2, 4)`, `2.bias (2,)` — and nothing from positions 1 and 3:
the gates own no numbers, exactly as Session 2 promised.
Keep the two vocabulary levels straight, because exam questions mix
them deliberately:

- **parameter tensors** — how many registered arrays (here: 4);
- **parameters** (scalar count) — how many individual numbers
  (Section 2 counts them: 12 + 4 + 8 + 2 = 26).

### Checkpoint 1

1. For `bench`: how many parameter *tensors*, and which children of
   the Sequential contribute none?
2. What does `named_parameters()` yield for a `TinyMLP`-style custom
   class (Session 2 §5) that stores the same two dense layers as
   attributes `hidden` and `readout`?
3. One `ThresholdGate` instance is reused three times inside a module's
   `forward`. How many parameter tensors does it contribute?



## 2. Counting by Hand: the Layer Formula

**Motivation.**
Parameter counts are exam arithmetic, done without running anything.
Everything follows from one line.

**A dense layer $\text{in} \to \text{out}$ stores**
$$\underbrace{\text{out} \times \text{in}}_{\text{weight}}
\;+\; \underbrace{\text{out}}_{\text{bias}}
\;=\; \text{out} \times (\text{in} + 1)$$
**numbers.**
The weight matrix has one row per output unit and one column per input;
the bias adds one more number per output unit — as if each unit had an
extra always-on input.

**A network of dense layers with unit counts
$n_0 \to n_1 \to \dots \to n_L$ stores**
$$P \;=\; \sum_{\ell=1}^{L} n_\ell\,(n_{\ell-1} + 1),$$
**and activation gates add zero** (they own no numbers — Section 1).

*Worked count 1 — the bench network $3 \to 4 \to 2$:*
layer 1: $4 \times (3 + 1) = 16$; layer 2: $2 \times (4 + 1) = 10$;
total $P = 26$.

*Worked count 2 — a deeper stack $5 \to 10 \to 10 \to 1$:*
$10 \times 6 = 60$, then $10 \times 11 = 110$, then
$1 \times 11 = 11$; total $P = 181$.
Note the middle layer is the expensive one — width talking to width.


In [ ]:

def hand_count(widths):
    """P = sum over layers of n_l * (n_{l-1} + 1), from the unit-count list."""
    return sum(o * (i + 1) for i, o in zip(widths[:-1], widths[1:]))


print("bench 3->4->2:      ", hand_count([3, 4, 2]))
print("stack 5->10->10->1: ", hand_count([5, 10, 10, 1]))



The helper agrees with both hand counts (26 and 181) — but understand
that on the exam *you* are the helper: the formula plus clean
arithmetic is the whole skill.

### Checkpoint 2

1. Count by hand: $8 \to 3$ single dense layer.
2. Count by hand: $2 \to 6 \to 6 \to 1$ (gates after each dense layer).
3. Which single layer of $20 \to 50 \to 30 \to 5$ holds the most
   parameters? Decide *before* multiplying everything out, then verify.



## 3. Counting in Code: `numel` — and Counting Without It

**Motivation.**
Every tensor reports its scalar size with **`.numel()`** ("number of
elements"), so the one-line audit of any module is a sum over
`parameters()`.


In [ ]:

total = sum(p.numel() for p in bench.parameters())
print("bench total parameters:", total)
print("matches the hand count:", total == hand_count([3, 4, 2]))

per_tensor = [(name, p.numel()) for name, p in bench.named_parameters()]
print("per tensor:", per_tensor)



`26`, matching Section 2, split as
`[('0.weight', 12), ('0.bias', 4), ('2.weight', 8), ('2.bias', 2)]`.

**The count-without-`numel` register.**
The exam's torch cluster likes to *ban the helper*: "compute the total
parameter count; `numel` is banned (zero points)."
The ban is not spite — it forces the Section 2 reasoning into code.
The sanctioned route reads each parameter's `.shape` and multiplies the
entries:


In [ ]:

total_no_numel = 0
for p in bench.parameters():
    size = 1
    for d in p.shape:                     # multiply out the shape tuple
        size *= d
    total_no_numel += size
print("count without numel:", total_no_numel)



Same `26`.
Under such a ban, treat the *whole helper family* as closed: workarounds
like `p.reshape(-1).shape[0]`, `len(p.flatten())`, or
`np.prod(p.shape)` are the same helper wearing sunglasses, and ban
clauses in this course (and rubrics on the exam) score them zero.
What the ban wants is visible: shapes in, arithmetic out.

### Checkpoint 3

1. Write the one-line `numel` audit for a module `m`, and the shape
   `(7, 2)` / `(7,)` layer's contribution it would report.
2. Under a "no `numel`" clause, why does `len(p.flatten())` still earn
   zero, and what is the sanctioned pattern?
3. `sum(1 for p in m.parameters())` computes something useful but *not*
   the parameter count. What?



## 4. `state_dict`: the Model as a Dictionary

**Motivation.**
The third member of the family, **`m.state_dict()`**, returns the
module's numbers as an ordered dictionary `{dotted_name: tensor}`.
It is torch's *save/load format* — when C7 downloads a pretrained
model, what arrives is exactly a `state_dict` — and it doubles as an
inspection tool and as a way to swap in new hand-set weights via
**`m.load_state_dict(...)`**.


In [ ]:

sd = bench.state_dict()
print("keys:", list(sd.keys()))
print("shape under '0.weight':", tuple(sd["0.weight"].shape))

# swap in new manual weights for the readout layer, by name:
new_sd = {name: tensor.clone() for name, tensor in sd.items()}
new_sd["2.weight"] = torch.ones(2, 4)
new_sd["2.bias"] = torch.tensor([-3.5, -0.5])
bench.load_state_dict(new_sd)

x = torch.tensor([[1.0, 1.0, 1.0]])
print("after load, readout weight:", bench[2].weight[0])
print("bench(x):", bench(x))



The keys mirror `named_parameters()` — `['0.weight', '0.bias',
'2.weight', '2.bias']` — and after `load_state_dict` the new readout
row prints `tensor([1., 1., 1., 1.])`.
The forward pass then reads: layer 1 is all zeros, so every
pre-activation is $0$, the gate turns all four to $1$ ($0 \ge 0$), and
the readout computes $4 - 3.5 = 0.5$ and $4 - 0.5 = 3.5$ — gated to
`tensor([[1., 1.]])`.

Three `state_dict` habits worth keeping:

- **read shapes from it** when handed a saved model: the widths
  reconstruct the architecture ($(4, 3)$ then $(2, 4)$ says
  $3 \to 4 \to 2$);
- **match names exactly** when loading: `load_state_dict` requires the
  dictionary's keys to agree with the module's registered names;
- **remember the ghost lesson**: only *registered* numbers appear.
  Session 2's plain-tensor layer saved an empty dictionary.

### Checkpoint 4

1. A saved `state_dict` has keys `['0.weight', '0.bias', '2.weight',
   '2.bias']` with shapes `(6, 2)`, `(6,)`, `(1, 6)`, `(1,)`.
   Reconstruct the architecture and the total parameter count.
2. After the cell above, compute `bench(torch.tensor([[0., 0., 0.]]))`
   by hand.
3. Which of the following appear in `state_dict()`:
   a registered `nn.Parameter` with `requires_grad=False`; a plain
   tensor attribute; a submodule's registered parameters?




## 5. Worked Exam-Style Example: Normal-Form MC on a Count

Parameter counting in the numeric normal form, solved in full.

---

**Problem (reasoning is not required; no code needed).**
A network maps $4 \to 6 \to 3$: a `zeros_dense(6, 4)`-shaped dense layer
(weight $(6, 4)$), gate, a `zeros_dense(3, 6)`-shaped one (weight
$(3, 6)$), gate, with biases throughout.
Let $F$ be the fraction of the network's parameters that live in the
**first** dense layer (weight and bias).
Write $F$ in lowest terms as $p/q$ with $\gcd(p, q) = 1$, $q > 0$.
What is $p + q$?

A. 11  B. 24  C. 25  D. 27  E. 81

---

**Solution.**

*Step 1 — the two layer counts.*
First layer: $6 \times (4 + 1) = 30$.
Second: $3 \times (6 + 1) = 21$.
Total $30 + 21 = 51$; gates add nothing.

*Step 2 — the fraction.*
$F = 30/51$; both divisible by 3: $F = 10/17$ in lowest terms.

*Step 3 — decode.*
$p + q = 10 + 17 = 27$: **answer D**.
The traps are built in: forgetting both biases gives $24/42 = 4/7
\mapsto 11$ (choice A); taking the *second* layer's share gives
$21/51 = 7/17 \mapsto 24$ (choice B); skipping the reduction of
$30/51$ gives $81$ (choice E); mixing a no-bias numerator with the
full denominator gives $24/51 = 8/17 \mapsto 25$ (choice C).

*Step 4 — the free cross-check (whenever code is allowed).*


In [ ]:

first = zeros_dense(6, 4)
second = zeros_dense(3, 6)
n_first = sum(p.numel() for p in first.parameters())
n_total = n_first + sum(p.numel() for p in second.parameters())
print(f"first layer {n_first}, total {n_total}  ->  F = {n_first}/{n_total} = 10/17, p + q = 27")



### Checkpoint 5

1. Redo the problem for $4 \to 6 \to 3$ *without any biases*:
   $F$, lowest terms, and $p + q$.
2. Which wrong habit produces each of choices A and E, and which
   single sentence of Section 2 kills both?



## 6. Backwards: from a Count to an Architecture

**Motivation.**
The count formula also runs in reverse — given a parameter budget and
an architecture family, Calc AB algebra pins the width.
This inverse direction is exactly the reasoning register of p12.

**Worked example.**
A network $2 \to h \to h \to 1$ (dense layers with biases, gates free)
reports **151 parameters**. Find $h$.

*Set up the count as a polynomial in $h$:*
$$P(h) = \underbrace{h(2+1)}_{3h}
\;+\; \underbrace{h(h+1)}_{h^2 + h}
\;+\; \underbrace{1 \cdot (h+1)}_{h + 1}
\;=\; h^2 + 5h + 1 .$$

*Solve $P(h) = 151$:*
$h^2 + 5h - 150 = 0$, which factors as $(h + 15)(h - 10) = 0$,
so $h = 10$ (the root $-15$ is not a width).

*Sanity-check the growth direction:* $P$ is an upward parabola in $h$
with positive coefficients, so it is strictly increasing for $h > 0$ —
the solution is unique, and roughly, doubling a middle width
quadruples its dominant $h^2$ term.
That quadratic dominance is why real networks' counts are ruled by
their width-to-width layers.


In [ ]:

h = 10
net = nn.Sequential(
    zeros_dense(h, 2), ThresholdGate(),
    zeros_dense(h, h), ThresholdGate(),
    zeros_dense(1, h), ThresholdGate(),
)
print("P(10) =", sum(p.numel() for p in net.parameters()))



`P(10) = 151` — the algebra and the module agree.

### Checkpoint 6

1. For the family $3 \to h \to 1$: write $P(h)$ and solve $P(h) = 41$.
2. Why can a strictly increasing $P$ never yield two candidate widths,
   and where did increasingness come from?
3. A teammate solves $P(h) = 151$ for the family above and reports
   $h = -15$ as "also valid mathematically." One-sentence verdict?



## 7. Common Pitfalls III

**Pitfall 1 — counting the gates.**
Activation modules are *layers* in the architectural sense but hold
**zero parameters**; padding the count with imagined "gate parameters"
is the most common hand-count error.
The audit is one line:


In [ ]:

lonely_gate = ThresholdGate()
print("gate parameter tensors:", len(list(lonely_gate.parameters())))
print("bench count, gates and all:", sum(p.numel() for p in bench.parameters()),
      "= hand count over dense layers only:", hand_count([3, 4, 2]))



`0` tensors for the gate, and the full `bench` count (26) equals the
dense-layers-only hand count — gates truly free.

**Pitfall 2 — forgetting the biases.**
$\text{out} \times \text{in}$ instead of
$\text{out} \times (\text{in} + 1)$: for `bench` that computes
$12 + 8 = 20$, off by the $4 + 2 = 6$ bias numbers.
Every trap option in Section 5 was built from this mistake.
When reading an unfamiliar module, do not guess — *ask the module*:
`[(n, tuple(p.shape)) for n, p in m.named_parameters()]` shows exactly
which bias tensors exist.

**Pitfall 3 — same count, wrong shapes.**
`numel` cannot distinguish a `(4, 3)` weight from a `(3, 4)` one —
both hold 12 numbers.
A transposed layer's *weight* numel is identical — and on a **square**
layer even the full totals agree — while it computes the wrong thing
(Session 2, Pitfall 1). In the non-square demo below the bias is the
only count-level tell: 16 vs 15, a close-but-not-equal total that
should smell like transposition.
Counting is a coarse audit; shape reading is the fine one:


In [ ]:

right_way = zeros_dense(4, 3)                       # weight (4, 3): 3 -> 4
wrong_way = zeros_dense(3, 4)                       # weight (3, 4): transposed layer
n_r = sum(p.numel() for p in right_way.parameters())
n_w = sum(p.numel() for p in wrong_way.parameters())
print("counts:", n_r, "vs", n_w, "- counts agree:", n_r == n_w)
print("shapes:", tuple(right_way.weight.shape), "vs", tuple(wrong_way.weight.shape))



The weight tensors hold the same 12 numbers each, yet the totals print
`16 vs 15` — transposition preserves the weight count
($4 \times 3 = 3 \times 4$) but changes the *bias* count ($4$ vs $3$).
Two lessons in one near-miss: a close-but-not-equal total is a
transposition smell, and on **square** layers even the totals agree,
leaving the count audit completely blind to the bug.
Read shapes, not just sums.

### Checkpoint 7

1. A classmate counts $2 \to 6 \to 1$ as
   $6 \times 2 + 1 \times 6 = 18$. What did they forget, and what is
   the correct count?
2. Two modules report equal `numel` totals. Does that make them the
   same architecture? Give the session's counterexample class.
3. Which inspection call settles "does layer 2 have a bias?" without
   reading any source code?



## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **PyTorch engineering cluster** (7 sub-parts, 50 points in
  r1-2026) is this unit's home ground: finishing `nn.Module`
  subclasses from a skeleton, wiring hand-set weights into small
  custom layers, predicting what a manual-weight MLP computes, and
  parameter arithmetic on given architectures.
  Notably, those problems are **inference-only** — like everything in
  this unit, they run networks forward with fixed weights and never
  adjust them.
- The register is exactly Session 2 §7's: exact class and method
  contracts, API bans with zero-point clauses (`nn.Linear` where
  hand-registration is the skill, `numel` where counting is), and
  "reasoning is required / not required" flags per part.
- Parameter counting appears both as calculator-free arithmetic in MC
  normal forms (Session 3 §5's register) and as shape reading on given
  modules; the analysis's difficulty profile places these among the
  paper's most bankable points *if* the layer formula is automatic.
- The C5→C6 pairing mirrors the paper's habit of asking for the same
  network twice — once as raw array math, once as a module — and
  scoring the agreement between them (p13's texture).

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C7-cnn-transfer`**: the modules stop being hand-sized. C7
  downloads a pretrained ResNet — a deep stack of modules whose
  `state_dict` arrives from the model zoo — and this unit's
  inspection tools (`named_parameters`, shape reading, counting)
  become the map you navigate it with, plus the surgery of swapping
  a layer for one of your own `DenseLayer` kind.
- The same `nn.Module` anatomy scales without change: a ResNet is
  `forward` plus registered submodules, exactly like `TinyMLP`, only
  deeper — reading C7's architectures is Session 2 read at scale.



## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Four tensors; the two `ThresholdGate` children (positions 1 and 3)
   contribute none.
2. The same four tensors under attribute-path names:
   `hidden.weight`, `hidden.bias`, `readout.weight`, `readout.bias`.
3. Zero — reuse in `forward` changes nothing; the gate registers no
   parameters however often it is called.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $3 \times (8 + 1) = 27$.
2. $6 \times 3 + 6 \times 7 + 1 \times 7 = 18 + 42 + 7 = 67$; the
   gates add nothing.
3. The $50 \to 30$ layer: width talking to width dominates
   ($30 \times 51 = 1530$ vs $50 \times 21 = 1050$ and
   $5 \times 31 = 155$).

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `sum(p.numel() for p in m.parameters())`; that layer contributes
   $14 + 7 = 21$.
2. `flatten` + `len` computes element count by another name — the ban
   covers the helper family, not one spelling.
   Sanctioned: read `p.shape` and multiply its entries.
3. The number of parameter *tensors* (Section 1's other vocabulary
   level), not the number of scalars.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $2 \to 6 \to 1$ with biases; count
   $6 \times 3 + 1 \times 7 = 25$ — matching
   $12 + 6 + 6 + 1 = 25$ from the shapes directly.
2. Layer 1 outputs zeros (zero weights), gate maps $0 \to 1$ on all
   four units, readout: $4 \cdot 1 - 3.5 = 0.5$ and
   $4 \cdot 1 - 0.5 = 3.5$, final gate: `[[1., 1.]]` — identical to
   the lesson's `bench(x)`, since layer 1 ignores its input entirely.
3. The frozen parameter: yes.  The plain tensor: no.  Submodule
   parameters: yes, under dotted names.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Counts become $24$ and $18$, total $42$: $F = 24/42 = 4/7$, so
   $p + q = 11$ — precisely trap A of the worked problem.
2. A: dropping biases; E: skipping the gcd reduction.
   The killer sentence: a dense layer holds
   $\text{out} \times (\text{in} + 1)$ numbers — the $+1$ *is* the
   bias, and normal forms demand lowest terms.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $P(h) = h(3 + 1) + (h + 1) = 5h + 1$; $5h + 1 = 41$ gives
   $h = 8$.
2. A strictly increasing function takes each value at most once;
   increasingness came from positive coefficients on $h$ and $h^2$ —
   more width always means more parameters.
3. A width is a number of units; $-15$ solves the equation but not
   the problem — discard non-positive roots.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. The biases — $6 + 1 = 7$ numbers are missing.
   Correct: $6 \times (2 + 1) + 1 \times (6 + 1) = 25$.
2. No — Pitfall 3's near-transposed pair (and any two same-budget
   architectures): equal totals, different shapes, different
   networks.
3. `m.named_parameters()` (or `m.state_dict().keys()`): a key
   `...bias` with the right prefix answers it directly.

</details>
